In [0]:
%pip install yfinance
%pip install lxml

In [0]:
import yfinance as yf
from delta.tables import DeltaTable

In [0]:
def rename(col):
    return col.replace("(","").replace("'","").replace(",","").replace(" ","").replace(")","").replace(f"{comp}","")

In [0]:
stoxx50_tickers = [
    "ABBN.SW",   # ABB (Switzerland)
    "AIR.PA",    # Airbus (France)
    "ALV.DE",    # Allianz (Germany)
    "ASML.AS",   # ASML (Netherlands)
    "AZN.L",     # AstraZeneca (UK)
    "BAS.DE",    # BASF (Germany)
    "BAYN.DE",   # Bayer (Germany)
    "BBVA.MC",   # BBVA (Spain)
    "BNP.PA",    # BNP Paribas (France)
    "BP.L",      # BP (UK)
    "BATS.L",    # British American Tobacco (UK)
    "DGE.L",     # Diageo (UK)
    "DTE.DE",    # Deutsche Telekom (Germany)
    "EL.PA",     # EssilorLuxottica (France)
    "GSK.L",     # GSK (UK)
    "GLEN.L",    # Glencore (UK)
    "RMS.PA",    # Hermes (France)
    "HSBA.L",    # HSBC (UK)
    "IBE.MC",    # Iberdrola (Spain)
    "ITX.MC",    # Inditex (Spain)
    "INGA.AS",   # ING Group (Netherlands)
    "ISP.MI",    # Intesa Sanpaolo (Italy)
    "KER.PA",    # Kering (France)
    "OR.PA",     # L'Oréal (France)
    "LIN.DE",    # Linde (Germany)
    "MC.PA",     # LVMH (France)
    "MBG.DE",    # Mercedes-Benz (Germany)
    "MUV2.DE",   # Munich Re (Germany)
    "NG.L",      # National Grid (UK)
    "NESN.SW",   # Nestlé (Switzerland)
    "NOVN.SW",   # Novartis (Switzerland)
    "NOVO-B.CO", # Novo Nordisk (Denmark)
    "PRU.L",     # Prudential (UK)
    "RKT.L",     # Reckitt Benckiser (UK)
    "REL.L",     # RELX (UK)
    "RIO.L",     # Rio Tinto (UK)
    "RO.SW",    # Roche (Switzerland)
    "SAF.PA",    # Safran (France)
    "SAN.PA",    # Sanofi (France)
    "SAP.DE",    # SAP (Germany)
    "SU.PA",     # Schneider Electric (France)
    "SHEL.L",    # Shell (UK)
    "SIE.DE",    # Siemens (Germany)
    "TTE.PA",    # TotalEnergies (France)
    "UBSG.SW",   # UBS Group (Switzerland)
    "ULVR.L",    # Unilever (UK)
    "DG.PA",     # Vinci (France)
    "VOD.L",     # Vodafone (UK)
    "VOW3.DE",   # Volkswagen Pref (Germany)
    "ZURN.SW"    # Zurich Insurance (Switzerland)
]

In [0]:
for comp in stoxx50_tickers:
    data = yf.download(comp, period="3y", auto_adjust=True)
    spark_df = spark.createDataFrame(data.reset_index())
    for col in spark_df.columns:
        spark_df = spark_df.withColumnRenamed(col, rename(col))

    tbl_name = comp.replace("-","_").replace(".","_")
    if not spark.catalog.tableExists(f"stocks.stage.{tbl_name}"):
           spark_df.write.mode("overwrite").format("delta").saveAsTable(f"stocks.stage.{tbl_name}"
            )

    delta_table = DeltaTable.forName(spark, f"stocks.stage.{tbl_name}")

    delta_table.alias("t").merge(
        spark_df.alias("s"),
        "s.Date = t.Date"
    ).whenNotMatchedInsertAll().execute()

In [0]:
# tables = [t.name for t in spark.catalog.listTables("stocks.bronze")]
# for table in tables:
#     spark.sql(f"DROP TABLE stocks.bronze.{table}")